In [22]:
#!/usr/bin/env python3
"""
MGMT 6130 - Assignment 4: Quantitative Risk Assessment Using I&W Analysis
Authors: Honglin Wang, Chuanli Yang, Raymond Twumasi Opoku
Microsoft Cybersecurity Acquisition Case Study

This notebook performs:
1. Indicators & Warning (I&W) quantitative analysis for three risk areas
2. Monte Carlo simulation for risk impact estimation
3. Machine Learning-based risk prediction validation
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report

np.random.seed(42)


# SECTION 1: Load and Prepare Risk Register Data

df = pd.read_excel('C:/Users/kevin/Desktop/New folder/files/risk_register.xlsx')
df.columns = df.columns.str.strip()
df = df.rename(columns={
    'Likelihood Score (1–15)': 'Likelihood_Score',
    'Impact Score (1–9+C+E4': 'Impact_Score'
})
df['Risk_Score'] = df['Likelihood_Score'] * df['Impact_Score']

# Assign risk area labels
df['Risk_Area'] = df['No.'].apply(lambda x: 
    'Market Share Loss' if x <= 15 else 
    'Data Breach (Integration)' if x <= 30 else 
    'Enterprise Acquisition (Bundled)')

print("=== Risk Register Summary ===")
print(f"Total risks: {len(df)}")
print(f"\nRisk Area Distribution:")
print(df['Risk_Area'].value_counts())
print(f"\nRisk Score Statistics by Area:")
print(df.groupby('Risk_Area')['Risk_Score'].describe().round(2))

=== Risk Register Summary ===
Total risks: 40

Risk Area Distribution:
Risk_Area
Market Share Loss                   15
Data Breach (Integration)           15
Enterprise Acquisition (Bundled)    10
Name: count, dtype: int64

Risk Score Statistics by Area:
                                  count   mean    std   min   25%   50%   75%  \
Risk_Area                                                                       
Data Breach (Integration)          15.0  75.80  14.93  49.0  68.0  72.0  90.0   
Enterprise Acquisition (Bundled)   10.0  57.20  21.38  28.0  42.0  54.0  70.0   
Market Share Loss                  15.0  66.53  17.52  32.0  54.0  70.0  78.5   

                                   max  
Risk_Area                               
Data Breach (Integration)         99.0  
Enterprise Acquisition (Bundled)  88.0  
Market Share Loss                 96.0  


In [23]:
# SECTION 2: I&W Indicator Definition and Threshold Calibration

"""
Indicators & Warning (I&W) Analysis Framework:
Each risk area is monitored via specific quantitative indicators.
Warning thresholds are calibrated using statistical analysis of risk scores.
"""

# Define I&W indicators for each risk area
iw_indicators = {
    'Market Share Loss': {
        'indicators': [
            'Customer churn rate (quarterly)',
            'Competitor product launch frequency',
            'Net Promoter Score (NPS) trend',
            'Market share % change (quarterly)',
            'R&D investment ratio vs competitors'
        ],
        'green_threshold': 'Risk Score < Q1 (score < 56)',
        'yellow_threshold': 'Q1 <= Risk Score < Q3 (56 <= score < 77)',
        'red_threshold': 'Risk Score >= Q3 (score >= 77)'
    },
    'Data Breach (Integration)': {
        'indicators': [
            'Unresolved security vulnerabilities count',
            'Failed penetration test findings',
            'Mean time to detect (MTTD) anomalies',
            'Unauthorized access attempt frequency',
            'Integration milestone completion rate'
        ],
        'green_threshold': 'Risk Score < Q1 (score < 64)',
        'yellow_threshold': 'Q1 <= Risk Score < Q3 (64 <= score < 90)',
        'red_threshold': 'Risk Score >= Q3 (score >= 90)'
    },
    'Enterprise Acquisition (Bundled)': {
        'indicators': [
            'Enterprise client pipeline conversion rate',
            'Onboarding capacity utilization %',
            'Bundle attach rate',
            'Client satisfaction score post-onboarding',
            'Revenue per enterprise client trend'
        ],
        'green_threshold': 'Risk Score < Q1 (score < 32)',
        'yellow_threshold': 'Q1 <= Risk Score < Q3 (32 <= score < 56)',
        'red_threshold': 'Risk Score >= Q3 (score >= 56)'
    }
}

# Compute actual thresholds from data
print("\n=== I&W Threshold Calibration by Risk Area ===")
for area in df['Risk_Area'].unique():
    subset = df[df['Risk_Area'] == area]['Risk_Score']
    q1, median, q3 = subset.quantile([0.25, 0.50, 0.75])
    print(f"\n{area}:")
    print(f"  Q1={q1:.0f}, Median={median:.0f}, Q3={q3:.0f}, Min={subset.min()}, Max={subset.max()}")
    print(f"  Green: < {q1:.0f}  |  Yellow: {q1:.0f}-{q3:.0f}  |  Red: >= {q3:.0f}")


=== I&W Threshold Calibration by Risk Area ===

Market Share Loss:
  Q1=54, Median=70, Q3=78, Min=32, Max=96
  Green: < 54  |  Yellow: 54-78  |  Red: >= 78

Data Breach (Integration):
  Q1=68, Median=72, Q3=90, Min=49, Max=99
  Green: < 68  |  Yellow: 68-90  |  Red: >= 90

Enterprise Acquisition (Bundled):
  Q1=42, Median=54, Q3=70, Min=28, Max=88
  Green: < 42  |  Yellow: 42-70  |  Red: >= 70


In [24]:
# SECTION 3: Monte Carlo Simulation for Each Risk Area

"""
Monte Carlo simulation models the uncertainty in risk outcomes.
For each risk area, we simulate 10,000 scenarios by sampling
likelihood and impact from distributions fitted to the observed data.
"""

n_simulations = 10000
mc_results = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, area in enumerate(df['Risk_Area'].unique()):
    subset = df[df['Risk_Area'] == area]
    
    # Fit distributions to observed likelihood and impact scores
    lik_mean, lik_std = subset['Likelihood_Score'].mean(), subset['Likelihood_Score'].std()
    imp_mean, imp_std = subset['Impact_Score'].mean(), subset['Impact_Score'].std()
    
    # Simulate using truncated normal distributions
    sim_likelihood = np.clip(
        np.random.normal(lik_mean, lik_std, n_simulations),
        subset['Likelihood_Score'].min(), subset['Likelihood_Score'].max()
    )
    sim_impact = np.clip(
        np.random.normal(imp_mean, imp_std, n_simulations),
        subset['Impact_Score'].min(), subset['Impact_Score'].max()
    )
    sim_risk_scores = sim_likelihood * sim_impact
    
    mc_results[area] = {
        'scores': sim_risk_scores,
        'mean': sim_risk_scores.mean(),
        'std': sim_risk_scores.std(),
        'p5': np.percentile(sim_risk_scores, 5),
        'p50': np.percentile(sim_risk_scores, 50),
        'p95': np.percentile(sim_risk_scores, 95),
        'prob_high': (sim_risk_scores >= 40).mean(),
        'VaR_95': np.percentile(sim_risk_scores, 95)
    }
    
    # Plot distribution
    ax = axes[idx]
    ax.hist(sim_risk_scores, bins=50, density=True, alpha=0.7, color=['#2196F3', '#F44336', '#4CAF50'][idx], edgecolor='white')
    ax.axvline(mc_results[area]['mean'], color='black', linestyle='--', linewidth=2, label=f"Mean={mc_results[area]['mean']:.1f}")
    ax.axvline(mc_results[area]['p95'], color='red', linestyle=':', linewidth=2, label=f"95th %ile={mc_results[area]['p95']:.1f}")
    ax.axvline(40, color='orange', linestyle='-', linewidth=2, label='High Threshold (40)')
    ax.set_title(f"{area}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Simulated Risk Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Monte Carlo Simulation: Risk Score Distributions (n=10,000)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/kevin/Desktop/New folder/files/monte_carlo_distributions.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n[Figure saved: monte_carlo_distributions.png]")

# Print Monte Carlo summary
print("\n=== Monte Carlo Simulation Results (10,000 iterations) ===")
for area, res in mc_results.items():
    print(f"\n{area}:")
    print(f"  Mean Risk Score: {res['mean']:.2f} (SD: {res['std']:.2f})")
    print(f"  5th Percentile:  {res['p5']:.2f}")
    print(f"  Median:          {res['p50']:.2f}")
    print(f"  95th Percentile: {res['p95']:.2f} (Value-at-Risk)")
    print(f"  P(High Risk):    {res['prob_high']*100:.1f}%")


[Figure saved: monte_carlo_distributions.png]

=== Monte Carlo Simulation Results (10,000 iterations) ===

Market Share Loss:
  Mean Risk Score: 65.20 (SD: 12.51)
  5th Percentile:  45.25
  Median:          64.85
  95th Percentile: 86.55 (Value-at-Risk)
  P(High Risk):    98.4%

Data Breach (Integration):
  Mean Risk Score: 74.58 (SD: 12.08)
  5th Percentile:  55.39
  Median:          74.07
  95th Percentile: 95.35 (Value-at-Risk)
  P(High Risk):    100.0%

Enterprise Acquisition (Bundled):
  Mean Risk Score: 55.37 (SD: 13.47)
  5th Percentile:  34.82
  Median:          54.59
  95th Percentile: 79.01 (Value-at-Risk)
  P(High Risk):    86.5%


In [25]:
# SECTION 4: I&W Composite Scoring Model

"""
Each indicator is scored 1-5 based on current intelligence.
The composite I&W score determines the warning level.
"""

# Simulate current indicator readings (hypothetical assessment)
iw_scores = {
    'Market Share Loss': {
        'Customer churn rate': 4,          # Elevated - competitors gaining
        'Competitor launches': 3,           # Moderate activity
        'NPS trend': 3,                     # Slight decline
        'Market share change': 4,           # Noticeable pressure
        'R&D ratio vs competitors': 3       # Slightly behind
    },
    'Data Breach (Integration)': {
        'Unresolved vulnerabilities': 5,    # Critical - 18 open findings
        'Failed pen test findings': 4,      # Several critical findings
        'MTTD anomalies': 4,                # Detection gaps during transition
        'Unauthorized access attempts': 3,  # Baseline activity
        'Integration milestone rate': 4     # Behind schedule
    },
    'Enterprise Acquisition (Bundled)': {
        'Pipeline conversion rate': 2,      # Strong conversion
        'Onboarding capacity': 3,           # Manageable strain
        'Bundle attach rate': 2,            # Good uptake
        'Client satisfaction': 2,           # Positive feedback
        'Revenue per client': 3             # Stable growth
    }
}

print("\n=== I&W Composite Scores ===")
fig, ax = plt.subplots(figsize=(10, 6))

area_names = []
composite_scores = []
colors = []

for area, indicators in iw_scores.items():
    scores = list(indicators.values())
    composite = np.mean(scores)
    composite_scores.append(composite)
    area_names.append(area)
    
    if composite >= 4.0:
        level = "RED (Critical)"
        colors.append('#F44336')
    elif composite >= 3.0:
        level = "YELLOW (Caution)"
        colors.append('#FFC107')
    else:
        level = "GREEN (Clear)"
        colors.append('#4CAF50')
    
    print(f"\n{area}:")
    for ind, score in indicators.items():
        print(f"  {ind}: {score}/5")
    print(f"  Composite I&W Score: {composite:.2f} → {level}")

bars = ax.barh(area_names, composite_scores, color=colors, edgecolor='black', height=0.5)
ax.set_xlabel('Composite I&W Score (1-5)', fontsize=12)
ax.set_title('I&W Analysis: Composite Warning Scores by Risk Area', fontsize=13, fontweight='bold')
ax.axvline(3.0, color='orange', linestyle='--', linewidth=1.5, label='Caution Threshold')
ax.axvline(4.0, color='red', linestyle='--', linewidth=1.5, label='Critical Threshold')
ax.set_xlim(0, 5.5)
for bar, score in zip(bars, composite_scores):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{score:.2f}', va='center', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('C:/Users/kevin/Desktop/New folder/files/iw_composite_scores.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n[Figure saved: iw_composite_scores.png]")


=== I&W Composite Scores ===

Market Share Loss:
  Customer churn rate: 4/5
  Competitor launches: 3/5
  NPS trend: 3/5
  Market share change: 4/5
  R&D ratio vs competitors: 3/5
  Composite I&W Score: 3.40 → YELLOW (Caution)

Data Breach (Integration):
  Unresolved vulnerabilities: 5/5
  Failed pen test findings: 4/5
  MTTD anomalies: 4/5
  Unauthorized access attempts: 3/5
  Integration milestone rate: 4/5
  Composite I&W Score: 4.00 → RED (Critical)

Enterprise Acquisition (Bundled):
  Pipeline conversion rate: 2/5
  Onboarding capacity: 3/5
  Bundle attach rate: 2/5
  Client satisfaction: 2/5
  Revenue per client: 3/5
  Composite I&W Score: 2.40 → GREEN (Clear)

[Figure saved: iw_composite_scores.png]


In [26]:
# SECTION 5: Sensitivity Analysis (Tornado Chart)

"""
Identify which indicators have the greatest influence on overall risk.
"""

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (area, indicators) in enumerate(iw_scores.items()):
    base_composite = np.mean(list(indicators.values()))
    sensitivities = {}
    
    for ind, base_val in indicators.items():
        # Low scenario: indicator drops by 1
        low_scores = list(indicators.values())
        low_scores[list(indicators.keys()).index(ind)] = max(1, base_val - 1)
        low_composite = np.mean(low_scores)
        
        # High scenario: indicator rises by 1
        high_scores = list(indicators.values())
        high_scores[list(indicators.keys()).index(ind)] = min(5, base_val + 1)
        high_composite = np.mean(high_scores)
        
        sensitivities[ind] = (low_composite - base_composite, high_composite - base_composite)
    
    names = list(sensitivities.keys())
    lows = [s[0] for s in sensitivities.values()]
    highs = [s[1] for s in sensitivities.values()]
    
    ax = axes[idx]
    y_pos = range(len(names))
    ax.barh(y_pos, highs, left=0, color='#F44336', alpha=0.7, label='+1 point')
    ax.barh(y_pos, lows, left=0, color='#2196F3', alpha=0.7, label='-1 point')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([n[:25] for n in names], fontsize=8)
    ax.set_title(f'{area}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Change in Composite Score')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.legend(fontsize=7)

plt.suptitle('Sensitivity Analysis: Impact of ±1 Point Change per Indicator', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/kevin/Desktop/New folder/files/sensitivity_tornado.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n[Figure saved: sensitivity_tornado.png]")


[Figure saved: sensitivity_tornado.png]


In [27]:
# SECTION 6: ML Validation - Random Forest with Monte Carlo Features

"""
Enhance the ML model with Monte Carlo-derived features.
"""

# Add Monte Carlo statistics as features
for area in df['Risk_Area'].unique():
    mask = df['Risk_Area'] == area
    df.loc[mask, 'MC_Mean'] = mc_results[area]['mean']
    df.loc[mask, 'MC_Std'] = mc_results[area]['std']
    df.loc[mask, 'MC_P95'] = mc_results[area]['p95']
    df.loc[mask, 'MC_Prob_High'] = mc_results[area]['prob_high']

# Target variable
def risk_tier(score):
    if score >= 40: return 'High'
    if score >= 15: return 'Medium'
    return 'Low'

df['Severity_Label'] = df['Risk_Score'].apply(risk_tier)

X = df[['Likelihood_Score', 'Impact_Score', 'MC_Mean', 'MC_Std', 'MC_Prob_High']]
y = df['Severity_Label']

model = RandomForestClassifier(
    n_estimators=50, max_depth=3, min_samples_leaf=5,
    random_state=60, class_weight='balanced'
)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=50)
y_pred = cross_val_predict(model, X, y, cv=skf)

print("\n=== Enhanced ML Classification Report (with MC features) ===")
print(classification_report(y, y_pred))

# Feature importance
model.fit(X, y)
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance['Feature'], importance['Importance'], color='#2196F3', edgecolor='black')
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importance (with Monte Carlo Features)', fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/kevin/Desktop/New folder/files/feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n[Figure saved: feature_importance.png]")


=== Enhanced ML Classification Report (with MC features) ===
              precision    recall  f1-score   support

        High       0.97      0.89      0.93        37
      Medium       0.33      0.67      0.44         3

    accuracy                           0.88        40
   macro avg       0.65      0.78      0.69        40
weighted avg       0.92      0.88      0.89        40


[Figure saved: feature_importance.png]


In [28]:
# SECTION 7: Summary Table for Report

print("\n=== FINAL QUANTITATIVE RISK SUMMARY ===")
print("="*80)
summary_data = []
for area in ['Market Share Loss', 'Data Breach (Integration)', 'Enterprise Acquisition (Bundled)']:
    subset = df[df['Risk_Area'] == area]
    mc = mc_results[area]
    iw_composite = np.mean(list(iw_scores[{
        'Market Share Loss': 'Market Share Loss',
        'Data Breach (Integration)': 'Data Breach (Integration)',
        'Enterprise Acquisition (Bundled)': 'Enterprise Acquisition (Bundled)'
    }[area]].values()))
    
    if iw_composite >= 4.0: iw_level = 'RED'
    elif iw_composite >= 3.0: iw_level = 'YELLOW'
    else: iw_level = 'GREEN'
    
    summary_data.append({
        'Risk Area': area,
        'Avg Risk Score': subset['Risk_Score'].mean(),
        'MC Mean': mc['mean'],
        'MC 95th %ile (VaR)': mc['p95'],
        'P(High)': f"{mc['prob_high']*100:.1f}%",
        'I&W Score': f"{iw_composite:.2f}",
        'Warning Level': iw_level
    })
    
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("="*80)
print("\nAll outputs generated successfully.")


=== FINAL QUANTITATIVE RISK SUMMARY ===
                       Risk Area  Avg Risk Score   MC Mean  MC 95th %ile (VaR) P(High) I&W Score Warning Level
               Market Share Loss       66.533333 65.197557           86.548816   98.4%      3.40        YELLOW
       Data Breach (Integration)       75.800000 74.580712           95.349500  100.0%      4.00           RED
Enterprise Acquisition (Bundled)       57.200000 55.368497           79.008412   86.5%      2.40         GREEN

All outputs generated successfully.
